# Solution C2 · The same model at different prevalences: the AUC stays, the precision collapses

**Type**: Break the model

## Objective

Measure what happens to the metrics of a **fixed** classifier when the fraction of positives in the test data drops from 50% to 0.5%. Derive the formula of the
precision as a function of recall, false positive rate and prevalence, and correct the probabilities for the change of prevalence.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, log_loss, precision_score, recall_score, roc_auc_score

X, y = make_classification(n_samples=80000, n_features=8, n_informative=5, n_redundant=0, class_sep=0.8, flip_y=0.02, random_state=0)
pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
m = LogisticRegression(max_iter=1000).fit(X[np.r_[pos[:3000], neg[:3000]]], y[np.r_[pos[:3000], neg[:3000]]])
pos_te, neg_te = pos[3000:], neg[3000:23000]                    # 20,000 test negatives


def generate_test(prevalence, seed=0):
    n_pos = int(round(prevalence / (1 - prevalence) * len(neg_te)))
    idx = np.r_[neg_te, np.random.default_rng(seed).choice(pos_te, n_pos, replace=False)]
    return X[idx], y[idx]


prevalences = [0.5, 0.2, 0.05, 0.01, 0.005]

## Commented solution

In [ ]:
rows = {}
for pi in prevalences:
    X_te, y_te = generate_test(pi)
    p = m.predict_proba(X_te)[:, 1]
    k = (pi / (1 - pi)) / (0.5 / 0.5)                                       # ratio between the odds of the new prevalence and those of training
    p_c = (p / (1 - p) * k) / (1 + p / (1 - p) * k)
    rows[pi] = {"auc": roc_auc_score(y_te, p), "avg_precision": average_precision_score(y_te, p),
                "accuracy": accuracy_score(y_te, p >= 0.5), "baseline": 1 - y_te.mean(),
                "precision": precision_score(y_te, p >= 0.5), "recall": recall_score(y_te, p >= 0.5),
                "mean_p": p.mean(), "logloss": log_loss(y_te, p),
                "corrected_mean_p": p_c.mean(), "corrected_logloss": log_loss(y_te, p_c), "corrected_recall": recall_score(y_te, p_c >= 0.5)}
table = pd.DataFrame(rows).T
table.round(3)

In [ ]:
X_b, y_b = generate_test(0.5)
p_b = m.predict_proba(X_b)[:, 1]
recall_b = ((p_b >= 0.5) & (y_b == 1)).sum() / (y_b == 1).sum()
fpr_b = ((p_b >= 0.5) & (y_b == 0)).sum() / (y_b == 0).sum()
theoretical_precision = pd.Series({pi: recall_b * pi / (recall_b * pi + fpr_b * (1 - pi)) for pi in prevalences})
print(pd.DataFrame({"theoretical": theoretical_precision, "measured": table["precision"]}).round(3).T.to_string())

In [ ]:
correction_cost = bool(table.loc[0.005, "corrected_recall"] < 0.1)
print("recall at prevalence 0.005 with threshold 0.5:", table.loc[0.005, "recall"].round(3), "-> after the prior correction:", table.loc[0.005, "corrected_recall"].round(3))

## Check

In [ ]:
with verify():
    check(list(table.columns[:8]) == ["auc", "avg_precision", "accuracy", "baseline", "precision", "recall", "mean_p", "logloss"], "first eight columns correct", "check names and order")
    check(table["auc"].max() - table["auc"].min() < 0.04 and table["recall"].max() - table["recall"].min() < 0.1, "AUC and recall are (almost) constant as the prevalence varies (they vary only by sampling noise)", "they are properties of the model")
    check(table.loc[0.01, "precision"] < 0.1 and table.loc[0.005, "precision"] < 0.05, "with prevalence 1% the precision is below 10%, with 0.5% below 5%", "precision at threshold 0.5")
    check((table.loc[[0.05, 0.01, 0.005], "accuracy"] < table.loc[[0.05, 0.01, 0.005], "baseline"]).all(), "below 5% prevalence the model is less accurate than someone who always says 'negative'", "accuracy against baseline")
    check(theoretical_precision is not None and close(theoretical_precision.reindex(table.index).to_numpy(), table["precision"].to_numpy(), atol=0.03), "the precision formula matches the measured one (gap < 0.03)", "R*pi / (R*pi + FPR*(1-pi)) with R and FPR measured at prevalence 0.5")
    check("corrected_mean_p" in table and close(table["corrected_mean_p"].to_numpy(), np.array(table.index), rtol=0.25), "after the correction the mean predicted probability is close to the prevalence (within 25%)", "odds * k / (1 + odds * k), with k = (pi/(1-pi))")
    check((table["corrected_logloss"].iloc[1:] < table["logloss"].iloc[1:]).all() and table.loc[0.005, "corrected_logloss"] < 0.1, "the log-loss improves at every prevalence below 50% (at 0.5% below 0.1)", "compare logloss and corrected_logloss")
    check(correction_cost is True and table.loc[0.005, "corrected_recall"] < 0.1 and table.loc[0.005, "corrected_recall"] < table.loc[0.005, "recall"], "threshold 0.5 on the corrected probabilities finds almost no positive: the threshold must be revised", "corrected_recall at prevalence 0.005")
done()

## What you learned

- **AUC and recall do not change** (they are conditional on the true class: they use only positives or only negatives), and the same goes for the false positive rate. The **precision changes
  enormously**: from 0.86 at 50% to 0.06 at 1%, because $P = \frac{R\pi}{R\pi + \mathrm{FPR}(1-\pi)}$ and with small $\pi$ the second term (false positives over a huge mass of negatives) dominates.
  It is the reason why with rare events you look at precision, average precision and the number of alarms per day, not the AUC.
- Below 5% prevalence the model is less "accurate" than someone who always says "no": accuracy is useless and must be replaced with metrics on the rare class.
- A model trained with balanced data (for convenience or with `class_weight`) gives probabilities that are **too high** in production: the predicted mean stays about 0.23, the true frequency is 0.005. The
  prior correction brings the probabilities to the right values (log-loss from 0.34 to 0.02) but **requires a new threshold**: with 0.5 on the corrected probabilities the recall is almost zero. Correcting is not
  an end in itself: you correct so that you can then choose the threshold with the real costs (B2).
- The sampling with which you build the test set is part of the evaluation model: a balanced test set "for convenience" describes a world that does not exist.

## Common pitfall

Evaluating a rare-event model (fraud, failures, diseases) on a balanced or rebalanced test set and reporting precision and accuracy as if they held in production. AUC and recall are
preserved, precision and accuracy are not: they must be computed on the **real prevalence**.

## Interview question

<details><summary>Your fraud model has AUC 0.92 but the antifraud team says 95% of the alarms are false. Is the model wrong?</summary>

Probably not: with a prevalence of 0.5% even an excellent classifier has low precision, because the precision depends on the prevalence (P = R·π / (R·π + FPR·(1−π))).
I check the precision on the real stream, compare it with the formula, and set the threshold based on how many alarms the team can work and the cost of an undetected fraud. The AUC says
the ordering is good; the precision says how much it costs to act on the top of the list.

</details>